In [1]:
import math                                                         # pour les calculs mathématiques
import json                                                         # pour afficher l'arbre proprement

# Dataset : [Outlook, Temperature, Humidity, Wind, Play?]
data = [
    ("Sunny",    "Hot",  "High",   "Weak",   "No"),               # jour 1
    ("Sunny",    "Hot",  "High",   "Strong", "No"),               # jour 2
    ("Overcast", "Hot",  "High",   "Weak",   "Yes"),              # jour 3
    ("Rain",     "Mild", "High",   "Weak",   "Yes"),              # jour 4
    ("Rain",     "Cool", "Normal", "Weak",   "Yes"),              # jour 5
    ("Rain",     "Cool", "Normal", "Strong", "No"),               # jour 6
    ("Overcast", "Cool", "Normal", "Strong", "Yes"),              # jour 7
    ("Sunny",    "Mild", "High",   "Weak",   "No"),               # jour 8
    ("Sunny",    "Cool", "Normal", "Weak",   "Yes"),              # jour 9
    ("Rain",     "Mild", "Normal", "Weak",   "Yes"),              # jour 10
    ("Sunny",    "Mild", "Normal", "Strong", "Yes"),              # jour 11
    ("Overcast", "Mild", "High",   "Strong", "Yes"),              # jour 12
    ("Overcast", "Hot",  "Normal", "Weak",   "Yes"),              # jour 13
    ("Rain",     "Mild", "High",   "Strong", "No"),               # jour 14
]

# ──────────────────────────────────────────────
# ÉTAPE 1 : Calculer l'indice de Gini d'un groupe
# Gini mesure le désordre : 0 = pur (tout Yes ou tout No), 0.5 = maximum de désordre
# Formule : Gini = 1 - somme(p_i²)
# C'est la différence principale avec ID3 qui utilise l'entropie
# ──────────────────────────────────────────────
def gini(dataset):
    total = len(dataset)                                            # nombre total de lignes
    if total == 0:                                                  # si le groupe est vide
        return 0                                                    # Gini = 0 par convention

    nbr_yes = sum(1 for row in dataset if row[4] == "Yes")         # compter les Yes
    nbr_no  = total - nbr_yes                                      # compter les No

    p_yes = nbr_yes / total                                        # probabilité d'avoir Yes
    p_no  = nbr_no  / total                                        # probabilité d'avoir No

    return 1 - (p_yes**2 + p_no**2)                               # formule de Gini

# ──────────────────────────────────────────────
# ÉTAPE 2 : Calculer le Gini d'un split (division en 2 groupes)
# CART divise toujours en exactement 2 groupes : col==valeur / col!=valeur
# C'est la différence principale avec ID3 qui divise en autant de groupes que de valeurs
# ──────────────────────────────────────────────
def gini_split(dataset, col, valeur):
    gauche = [row for row in dataset if row[col] == valeur]        # groupe gauche : col == valeur
    droite = [row for row in dataset if row[col] != valeur]        # groupe droite : col != valeur

    total  = len(dataset)                                          # taille totale

    poids_gauche = len(gauche) / total                             # poids du groupe gauche
    poids_droite = len(droite) / total                             # poids du groupe droite

    gini_split = poids_gauche * gini(gauche) + poids_droite * gini(droite)  # Gini pondéré des 2 groupes

    return gini_split                                              # retourner le Gini de ce split

# ──────────────────────────────────────────────
# ÉTAPE 3 : Trouver le meilleur split (meilleure colonne + meilleure valeur)
# On teste TOUTES les combinaisons colonne/valeur et on garde celle avec le Gini le plus bas
# ──────────────────────────────────────────────
def meilleur_split(dataset, attributs):
    meilleur_gini = float('inf')                                   # on commence avec un Gini infini
    meilleure_col = None                                           # colonne du meilleur split
    meilleure_val = None                                           # valeur du meilleur split

    for col in attributs:                                          # pour chaque colonne disponible
        valeurs = set(row[col] for row in dataset)                 # toutes les valeurs uniques de cette colonne
        for val in valeurs:                                        # pour chaque valeur possible
            g = gini_split(dataset, col, val)                      # calculer le Gini de ce split
            print(f"  Gini(col={col}, val={val}) = {g:.4f}")      # afficher pour visualiser
            if g < meilleur_gini:                                  # si c'est le meilleur jusqu'ici
                meilleur_gini = g                                  # sauvegarder le Gini
                meilleure_col = col                                # sauvegarder la colonne
                meilleure_val = val                                # sauvegarder la valeur

    return meilleure_col, meilleure_val                            # retourner le meilleur split trouvé

# ──────────────────────────────────────────────
# ÉTAPE 4 : Classe majoritaire d'un groupe
# Utilisée quand on arrive à une feuille : on retourne Yes ou No selon qui est majoritaire
# ──────────────────────────────────────────────
def classe_majoritaire(dataset):
    nbr_yes = sum(1 for row in dataset if row[4] == "Yes")         # compter les Yes
    nbr_no  = len(dataset) - nbr_yes                               # compter les No
    return "Yes" if nbr_yes >= nbr_no else "No"                    # retourner la classe majoritaire

# ──────────────────────────────────────────────
# ÉTAPE 5 : Algorithme CART (Classification And Regression Tree)
# Construit un arbre binaire récursivement en cherchant le meilleur split à chaque noeud
# ──────────────────────────────────────────────
def cart(dataset, attributs, profondeur_max=3, profondeur=0):

    # ── Cas 1 : tout Yes → feuille pure
    if all(row[4] == "Yes" for row in dataset):                    # si tous les exemples sont Yes
        return "Yes"                                               # feuille = Yes

    # ── Cas 2 : tout No → feuille pure
    if all(row[4] == "No" for row in dataset):                     # si tous les exemples sont No
        return "No"                                                # feuille = No

    # ── Cas 3 : profondeur max atteinte → classe majoritaire
    if profondeur >= profondeur_max:                               # si on a atteint la profondeur limite
        return classe_majoritaire(dataset)                         # retourner la classe majoritaire

    # ── Cas 4 : plus d'attributs → classe majoritaire
    if len(attributs) == 0:                                        # si plus aucun attribut disponible
        return classe_majoritaire(dataset)                         # retourner la classe majoritaire

    # ── Cas 5 : choisir le meilleur split
    print(f"\n── Profondeur {profondeur} ──")                      # afficher la profondeur actuelle
    col, val = meilleur_split(dataset, attributs)                  # trouver le meilleur split

    # Séparer en 2 groupes selon le split trouvé
    gauche = [row for row in dataset if row[col] == val]           # groupe gauche : col == val
    droite = [row for row in dataset if row[col] != val]           # groupe droite : col != val

    # Créer le noeud de l'arbre avec la condition du split
    noeud = {f"col{col}=={val}": {}}                               # clé = la condition du split

    # Appel récursif sur le groupe gauche (col == val)
    noeud[f"col{col}=={val}"]["oui"] = cart(gauche, attributs, profondeur_max, profondeur + 1)

    # Appel récursif sur le groupe droite (col != val)
    noeud[f"col{col}=={val}"]["non"] = cart(droite, attributs, profondeur_max, profondeur + 1)

    return noeud                                                   # retourner le noeud complet

# ──────────────────────────────────────────────
# MAIN
# ──────────────────────────────────────────────
print("Gini du dataset complet :", round(gini(data), 4))           # afficher le Gini initial

attributs = [0, 1, 2, 3]                                          # Outlook, Temperature, Humidity, Wind

arbre = cart(data, attributs, profondeur_max=3)                    # construire l'arbre CART

print("\n── Arbre CART ──")
print(json.dumps(arbre, indent=4))                                 # afficher l'arbre final

Gini du dataset complet : 0.4592

── Profondeur 0 ──
  Gini(col=0, val=Overcast) = 0.3571
  Gini(col=0, val=Rain) = 0.4571
  Gini(col=0, val=Sunny) = 0.3937
  Gini(col=1, val=Hot) = 0.4429
  Gini(col=1, val=Cool) = 0.4500
  Gini(col=1, val=Mild) = 0.4583
  Gini(col=2, val=Normal) = 0.3673
  Gini(col=2, val=High) = 0.3673
  Gini(col=3, val=Strong) = 0.4286
  Gini(col=3, val=Weak) = 0.4286

── Profondeur 1 ──
  Gini(col=0, val=Rain) = 0.4800
  Gini(col=0, val=Sunny) = 0.4800
  Gini(col=1, val=Hot) = 0.3750
  Gini(col=1, val=Cool) = 0.4762
  Gini(col=1, val=Mild) = 0.4800
  Gini(col=2, val=Normal) = 0.3200
  Gini(col=2, val=High) = 0.3200
  Gini(col=3, val=Strong) = 0.4167
  Gini(col=3, val=Weak) = 0.4167

── Profondeur 2 ──
  Gini(col=0, val=Rain) = 0.2667
  Gini(col=0, val=Sunny) = 0.2667
  Gini(col=1, val=Cool) = 0.2667
  Gini(col=1, val=Mild) = 0.2667
  Gini(col=2, val=Normal) = 0.3200
  Gini(col=3, val=Strong) = 0.2000
  Gini(col=3, val=Weak) = 0.2000

── Profondeur 2 ──
  Gini(col=0